In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


# OneTrainer on Google Colab


## Check GPU and Setup Environment


In [ ]:

!nvidia-smi

import sys
print(f"python {sys.version.split()[0]}")
print(f"python {sys.version_info[:3]}")

if sys.version_info < (3, 10) or sys.version_info >= (3, 13):
    print("onetrainer wants python >=3.10,<3.13")
else:
    print("python ok")


Mon Aug 24 22:19:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   38C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# @title
# # @title
# %cd {ROOT}/OneTrainer/src/diffusers
# !git status
# !git stash -u
# print('splug')
# %cd {ROOT}/OneTrainer
# # print('splog')
# !pip install -r requirements.txt


In [ ]:
import os
!git clone https://github.com/huggingface/diffusers.git /content/diffusers

%cd /content/diffusers
!git status
print('splug')

if not os.path.exists('/content/OneTrainer'):
    !git clone https://github.com/Nerogar/OneTrainer.git /content/OneTrainer

%cd /content/OneTrainer
!pip install -r requirements.txt


Cloning into '/content/diffusers'...
remote: Enumerating objects: 131571, done.
remote: Counting objects: 100% (1878/1878), done.
remote: Compressing objects: 100% (1110/1110), done.
remote: Total 131571 (delta 1374), reused 768 (delta 768), pack-reused 129693 (from 4)
Receiving objects: 100% (131571/131571), 106.59 MiB | 11.36 MiB/s, done.
Resolving deltas: 100% (97928/97928), done.
/content/diffusers
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
splug
Cloning into '/content/OneTrainer'...
remote: Enumerating objects: 17174, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 17174 (delta 121), reused 62 (delta 62), pack-reused 17004 (from 2)
Receiving objects: 100% (17174/17174), 5.36 MiB | 12.62 MiB/s, done.
Resolving deltas: 100% (13964/13964), done.
/content/OneTrainer
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
Ignoring t

In [2]:
%cd {ROOT}/OneTrainer

import subprocess
print(subprocess.run(['grep','-rn','write_video','modules/','scripts/'],
                     capture_output=True, text=True).stdout)


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


KeyboardInterrupt: 

In [4]:
# !python scripts/create_train_files.py -h

%cd {ROOT}/OneTrainer

# !python scripts/create_train_files.py \
#   --config-output-destination {ROOT}/training/configs/superseded/train_config.json \
#   --concepts-output-destination {ROOT}/training/configs/superseded/train_concepts.json \
#   --samples-output-destination {ROOT}/training/configs/superseded/train_samples.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


In [ ]:
!python - <<'PY'
import json
cfg=str(ROOT / "train_config.json")
with open(cfg) as f: c=json.load(f)
print("concept_file_name:", c.get("concept_file_name"))
print("sample_definition_file_name:", c.get("sample_definition_file_name"))
print("output_model_destination:", c.get("output_model_destination"))
print("workspace_dir:", c.get("workspace_dir"))
print("cache_dir:", c.get("cache_dir"))
print("base_model_name:", c.get("base_model_name"))
print("model_type:", c.get("model_type"), "peft_type:", c.get("peft_type"))
print("epochs:", c.get("epochs"), "batch_size:", c.get("batch_size"))


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
concept_file_name: /content/drive/MyDrive/Synthetic_Plants_Project/train_concepts.json
sample_definition_file_name: /content/drive/MyDrive/Synthetic_Plants_Project/train_samples.json
output_model_destination: /content/drive/MyDrive/Synthetic_Plants_Project/outputs/achillea/model.safetensors
workspace_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace/achillea_run
cache_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace-cache/achillea_run
base_model_name: stable-diffusion-v1-5/stable-diffusion-v1-5
model_type: STABLE_DIFFUSION_15 peft_type: LORA
epochs: 100 batch_size: 1


In [10]:
!python -c "import site, sys; print(site.getsitepackages()); print([p for p in sys.path if 'packages' in p])"
!ls -la /usr/local/lib/python3.13/dist-packages/sitecustomize.py


['/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/lib/python3.13/dist-packages']
['/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages']
-rw-r--r-- 1 root root 123 Aug 24 22:43 /usr/local/lib/python3.13/dist-packages/sitecustomize.py


In [12]:
!python -c "import sitecustomize" 2>&1 | tail -20


In [13]:
!python -c "import torchvision.io; print(hasattr(torchvision.io,'write_video'))"


False


In [2]:
%%writefile /content/shim.py
import sys, os, runpy
import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None

root = os.getcwd()
sys.path.insert(0, os.path.join(root, "scripts"))
sys.path.insert(0, root)

sys.argv = sys.argv[1:]
runpy.run_path(os.path.join(root, "scripts", "train.py"), run_name="__main__")


Overwriting /content/shim.py


In [19]:
!grep -i diffusers {ROOT}/OneTrainer/requirements*.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:-e git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers


In [2]:
!grep -iE "transformers|torchvision|torch==|accelerate" {ROOT}/OneTrainer/requirements*.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-cuda.txt:torch==2.8.0+cu128
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-cuda.txt:torchvision==0.23.0+cu128
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-default.txt:torch==2.8.0
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-default.txt:torchvision==0.23.0
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:accelerate==1.7.0
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:transformers==4.56.2
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:sentencepiece==0.2.1 # transitive dependency of transformers for tokenizer loading
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:open-clip-torch==2.32.0
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer/requirements-global.txt:lion-pytorch==0.2.3 # lion opti

In [3]:
!pip install torch==2.8.0 torchvision==0.23.0 transformers==4.56.2 accelerate==1.7.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 151.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.7.0
    Uninstalling triton-3.7.0:
      Successfully uninstalled triton-3.7.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.28.9
    Uninstalling nvidia-nccl-

In [20]:
!pip install "git+https://github.com/huggingface/diffusers.git@6a1904e"


  Cloning https://github.com/huggingface/diffusers.git (to revision 6a1904e) to /tmp/pip-req-build-mie01zq2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-mie01zq2
  Running command git checkout -q 6a1904e
  Resolved https://github.com/huggingface/diffusers.git to commit 6a1904e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.37.0.dev0-py3-none-any.whl size=4917914 sha256=05692f1fffdb5b1706d2d0c0245b29ac56f94eb5d0d88ab5aebafbde3a8df5a0
  Stored in directory: /tmp/pip-ephem-wheel-cache-qx0f3vy2/wheels/c6/6a/99/0c9c0412ded58c19bf1fa5d9df76bac26c2f2aac1f9826c20c
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.39.0.dev0
    Uninstalling diffusers-0.39.0.dev0:
      Successfully uninstalled diffusers-0.39.0.dev0


In [5]:
%cd {ROOT}/OneTrainer
!git pull
!git submodule update --init --recursive
!pip install -r requirements.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
Updating 9a270603..23df3832
error: Your local changes to the following files would be overwritten by merge:
	modules/modelSampler/StableDiffusionSampler.py
	run-cmd.sh
	start-ui.sh
Please commit your changes or stash them before you merge.
Aborting
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Obtaining diffusers from git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers (from -r requirements-global.txt (line 23))
  Updating ./src/diffusers clone (to revision 6a1904e)
  Running command git fetch -q --tags
  Running command git reset --hard -q 6a1904e
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining mgds from git+https://g

In [4]:
!python -c "import mgds; print(mgds.__file__)"
!cd {ROOT}/OneTrainer && git log -1 --format=%cd && git submodule status
!cd /content/OneTrainer && git log -1 --format=%cd


None
Sun Feb 8 09:18:39 2026 +0100
Wed Aug 19 21:38:17 2026 +0200


In [3]:
%cd {ROOT}/OneTrainer
!python /content/shim.py train.py --config-path {ROOT}/training/configs/qwen/qwen_finetune_preset.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
2026-08-24 22:55:11.666590: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 22:55:11.737078: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Traceback (most r

In [6]:
# !python scripts/convert_model.py -h

!python scripts/create_train_files.py -h


usage: create_train_files.py [-h]
                             [--config-output-destination CONFIG_OUTPUT_DESTINATION]
                             [--concepts-output-destination CONCEPTS_OUTPUT_DESTINATION]
                             [--samples-output-destination SAMPLES_OUTPUT_DESTINATION]

One Trainer Create Train Files Script.

options:
  -h, --help            show this help message and exit
  --config-output-destination CONFIG_OUTPUT_DESTINATION
                        The destination filename to save a default config file
  --concepts-output-destination CONCEPTS_OUTPUT_DESTINATION
                        The destination filename to save a default concepts
                        file
  --samples-output-destination SAMPLES_OUTPUT_DESTINATION
                        The destination filename to save a default samples
                        file


In [7]:
%%writefile /usr/local/lib/python3.13/dist-packages/sitecustomize.py
import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None


Overwriting /usr/local/lib/python3.13/dist-packages/sitecustomize.py


In [11]:
!python -c "import torchvision.io; print(hasattr(torchvision.io,'write_video'))"


False


In [9]:
%cd {ROOT}/OneTrainer
!python scripts/train.py --config-path {ROOT}/training/configs/qwen/qwen_finetune_preset.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
  File "/usr/local/lib/python3.13/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
  File "/usr/lib/python3.13/ctypes/__init__.py", line 482, in LoadLibrary
    return self._dlltype(name)
           ~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 361, in __init__
    self._handle = self._load_library(name, mode, handle, winmode)
                   ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 403, in _load_library
    return _dlopen(name, mode)
OSError: libnvJitLink.so.13: cannot open shared object file:

In [ ]:
# !python scripts/train.py \
#   --config-path {ROOT}/training/configs/qwen/qwen_finetune_preset.json \


2026-02-12 15:59:28.944896: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 15:59:28.963752: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770911968.987514    6161 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770911968.995238    6161 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770911969.015832    6161 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 